[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AI-Hypercomputer/maxtext/blob/main/src/maxtext/examples/native_lora_demo.ipynb)

# Parameter-Efficient Fine-Tuning (PEFT) Demo with Native LoRA and QLoRA

## Overview

This tutorial demonstrates how to run native Parameter-Efficient Fine-Tuning (PEFT) using **LoRA** and **QLoRA** (base weight quantization such as `int8`, `nf4`, or `fp8`) in MaxText across supported models (such as **Gemma4** and **Qwen3**).

We cover two workflows:
1. **Native LoRA / QLoRA Supervised Fine-Tuning (SFT)** on the GSM8K dataset, starting from converted Hugging Face base checkpoints.
2. **Native Pre-training with QLoRA** using memory-efficient quantized base weights.

## Prerequisites

Before running this notebook, make sure your environment is set up for the method you are using. Follow the [Run MaxText Python Notebooks on TPUs](https://maxtext.readthedocs.io/en/latest/guides/run_python_notebook.html) guide and complete all steps for your chosen method (Google Colab, VS Code, or Local Jupyter Lab) before proceeding.

If you run into issues, refer to the [Common Pitfalls & Debugging](https://maxtext.readthedocs.io/en/latest/guides/run_python_notebook.html#common-pitfalls-debugging) section of the guide.

In [ ]:
try:
  import google.colab
  print("Running the notebook on Google Colab")
  IN_COLAB = True
except ImportError:
  print("Running the notebook on Visual Studio or JupyterLab")
  IN_COLAB = False

## Installation: MaxText and Post-Training Dependencies

**Running the notebook on Visual Studio or JupyterLab:** Before proceeding, create a virtual environment and install the required post-training dependencies by following `Option 3: Installing [tpu-post-train]` in the [MaxText installation guide](https://maxtext.readthedocs.io/en/latest/install_maxtext.html#from-source). Once the environment is set up, ensure the notebook is running within it.

In [ ]:
if IN_COLAB:
    # Clone the MaxText repository
    !git clone https://github.com/AI-Hypercomputer/maxtext.git
    %cd maxtext

    # Install uv, a fast Python package installer
    !pip install uv
    
    # Install MaxText and post-training dependencies
    import os
    os.environ["UV_TORCH_BACKEND"]="cpu"
    !uv pip install -e .[tpu-post-train] --resolution=lowest
    !install_tpu_post_train_extra_deps

**Session restart Instructions for Colab:**
1.  Navigate to the menu at the top of the screen.
2.  Click on **Runtime**.
3.  Select **Restart session** from the dropdown menu.

You will be asked to confirm the action in a pop-up dialog. Click on **Yes**.

## Setup and Imports

In [ ]:
import os
import sys
import subprocess
from etils import epath

## Model Configurations

Select your target model family below (`gemma4-e2b` or `qwen3-0.6b`). All checkpoint directories, run names, and training configurations are dynamically derived from the model selection.

In [ ]:
# Choose model: "gemma4-e2b" or "qwen3-0.6b"
MODEL_NAME = "gemma4-e2b"  # @param ["gemma4-e2b", "qwen3-0.6b"]
SCAN_LAYERS = False if MODEL_NAME.startswith("gemma") else True

# Output directories and checkpoint paths derived dynamically from model name
BASE_OUTPUT_DIRECTORY = f"/tmp/{MODEL_NAME}_output"
MODEL_CHECKPOINT_PATH = f"{BASE_OUTPUT_DIRECTORY}/{MODEL_NAME}_checkpoint"
ORBAX_ITEMS_PATH = os.path.join(MODEL_CHECKPOINT_PATH, "0/items")
PRETRAIN_OUTPUT_DIRECTORY = f"/tmp/{MODEL_NAME}_qlora_checkpoint"

print(f"Selected model: {MODEL_NAME}")
print(f"Scan layers: {SCAN_LAYERS}")
print(f"Base output directory: {BASE_OUTPUT_DIRECTORY}")
print(f"Model checkpoint path: {ORBAX_ITEMS_PATH}")

## Hugging Face Hub Login

Model checkpoints are hosted on the Hugging Face Hub. Run the cell below to log in so that you can download the base model weights.

In [ ]:
if IN_COLAB:
    from huggingface_hub import notebook_login
    notebook_login()
else:
    from huggingface_hub import login
    login()

## Download & Convert Checkpoint from Hugging Face

We convert the base Hugging Face checkpoint to the MaxText Orbax format using the standalone checkpoint conversion tool.

In [ ]:
if not epath.Path(ORBAX_ITEMS_PATH).exists():
    print(f"Converting checkpoint for {MODEL_NAME} from Hugging Face...")
    env = os.environ.copy()
    env["JAX_PLATFORMS"] = "cpu"
    subprocess.run([
        sys.executable, "-m", "maxtext.checkpoint_conversion.to_maxtext",
        f"model_name={MODEL_NAME}",
        f"base_output_directory={MODEL_CHECKPOINT_PATH}",
        "use_multimodal=false",
        f"scan_layers={SCAN_LAYERS}",
        "skip_jax_distributed_system=True",
    ], env=env, check=True)
    print(f"Checkpoint successfully converted to: {ORBAX_ITEMS_PATH}")
else:
    print(f"Model checkpoint already exists at: {ORBAX_ITEMS_PATH}")

## Native LoRA / QLoRA Supervised Fine-Tuning (SFT)

We run native Supervised Fine-Tuning (SFT) on the real `openai/gsm8k` math dataset with LoRA / QLoRA enabled.

We enable LoRA by passing `lora.enable_lora=True` (and optionally `lora.lora_weight_qtype=int8` or `nf4` for quantized weights), ensuring that only lightweight adapter parameters are trained while the base weights remain frozen.

In [ ]:
subprocess.run([
    sys.executable, "-m", "maxtext.trainers.post_train.sft.train_sft_native",
    f"run_name=lora_{MODEL_NAME}_sft_demo",
    f"load_parameters_path={ORBAX_ITEMS_PATH}",
    f"model_name={MODEL_NAME}",
    f"base_output_directory={BASE_OUTPUT_DIRECTORY}",
    f"scan_layers={SCAN_LAYERS}",
    "hf_path=openai/gsm8k",
    "train_split=train",
    "hf_data_dir=main",
    "train_data_columns=['question', 'answer']",
    "steps=5",
    "per_device_batch_size=1",
    "max_target_length=128",
    "learning_rate=3e-6",
    "weight_dtype=bfloat16",
    "dtype=bfloat16",
    "formatting_func_path=maxtext.input_pipeline.instruction_data_processing.math_qa_formatting",
    "formatting_func_kwargs={'template_path': 'src/maxtext/examples/chat_templates/math_qa.json'}",
    "lora.enable_lora=True",
    "lora.lora_weight_qtype=int8",
    "lora.lora_tile_size=32",
    "lora.lora_rank=4",
    "lora.lora_alpha=8.0",
], check=True)

## Native Pre-training with QLoRA (Quantized Base Weights)

Next, we demonstrate how to run a native pre-training loop with memory-efficient **QLoRA** enabled (`lora.enable_lora=True`, `lora.lora_weight_qtype=int8`).

In [ ]:
subprocess.run([
    sys.executable, "-m", "maxtext.trainers.pre_train.train",
    f"run_name=qlora_{MODEL_NAME}_pretrain_demo",
    f"model_name={MODEL_NAME}",
    f"scan_layers={SCAN_LAYERS}",
    "steps=1",
    "dataset_type=synthetic",
    "per_device_batch_size=1",
    "max_target_length=32",
    "enable_checkpointing=True",
    "checkpoint_period=1",
    "async_checkpointing=False",
    "sharding_tolerance=1.0",
    f"base_output_directory={PRETRAIN_OUTPUT_DIRECTORY}",
    "attention=dot_product",
    "weight_dtype=bfloat16",
    "dtype=bfloat16",
    "lora.enable_lora=True",
    "lora.lora_weight_qtype=int8",
    "lora.lora_tile_size=32",
    "lora.lora_rank=4",
    "lora.lora_alpha=8.0",
], check=True)

## Conclusion

This tutorial successfully demonstrated how to run native Parameter-Efficient Fine-Tuning (PEFT) with LoRA and QLoRA in MaxText.

## 📚 Learn More

- **CLI Usage**: Refer to [Native LoRA and QLoRA Fine-tuning](https://github.com/AI-Hypercomputer/maxtext/blob/main/docs/tutorials/posttraining/native_lora.md) for detailed CLI orchestrations.
- **Configuration**: See `src/maxtext/configs/post_train/sft.yml` and `src/maxtext/configs/base.yml` for all available LoRA options (prefixed with `lora.`).
- **Documentation**: Check `src/maxtext/trainers/post_train/sft/train_sft_native.py` and `src/maxtext/trainers/pre_train/train.py` for native training implementations.